In [2]:
# =========================================================
# FULL CNN + EVIDENTIAL + DEEP ENSEMBLE (CPU, JUPYTER)
# =========================================================

# ---------- IMPORTS ----------
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ---------- PATHS ----------
DATA_DIR = r"D:\PROJECT -MTech\Datasets_SARCASM\MALAYALAM"
RESULT_DIR = r"D:\PROJECT -MTech\S4\results\evidential_deep\lstm"
os.makedirs(RESULT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_mal_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_mal_dev.csv")

DEVICE = torch.device("cpu")

# ---------- SEED ----------
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# ---------- LOAD DATA ----------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)

# ---------- LABEL CLEANING (CORRECT FOR YOUR DATASET) ----------
# labels: "Non-sarcastic" / "Sarcastic"
train_df['labels'] = train_df['labels'].astype(str).str.strip().str.lower()
dev_df['labels']   = dev_df['labels'].astype(str).str.strip().str.lower()

label_map = {
    "non-sarcastic": 0,
    "sarcastic": 1
}

train_df['labels'] = train_df['labels'].map(label_map)
dev_df['labels']   = dev_df['labels'].map(label_map)

# drop invalid rows
train_df = train_df.dropna(subset=['labels'])
dev_df   = dev_df.dropna(subset=['labels'])

train_df['labels'] = train_df['labels'].astype(int)
dev_df['labels']   = dev_df['labels'].astype(int)

print("Train label distribution:")
print(train_df['labels'].value_counts())
print("Train size:", len(train_df))
print("Dev size:", len(dev_df))

# ---------- TOKENIZER ----------
MAX_VOCAB = 20000
MAX_LEN   = 50

from collections import Counter

def build_vocab(texts):
    counter = Counter()
    for t in texts:
        counter.update(str(t).split())
    vocab = {w: i+2 for i,(w,_) in enumerate(counter.most_common(MAX_VOCAB))}
    vocab["<PAD>"] = 0
    vocab["<UNK>"] = 1
    return vocab

vocab = build_vocab(train_df['Text'])
VOCAB_SIZE = len(vocab)

def encode(text):
    tokens = str(text).split()
    ids = [vocab.get(tok, 1) for tok in tokens][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

# ---------- DATASET ----------
class SarcasmDataset(Dataset):
    def __init__(self, df):
        self.texts = df['Text'].values
        self.labels = df['labels'].values

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(encode(self.texts[idx]), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

train_loader = DataLoader(
    SarcasmDataset(train_df),
    batch_size=32,
    shuffle=True
)

dev_loader = DataLoader(
    SarcasmDataset(dev_df),
    batch_size=32,
    shuffle=False
)

# ---------- MODEL ----------
class EvidentialLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)          # (B, L, E)
        _, (h_n, _) = self.lstm(x)     # h_n: (1, B, H)
        h = h_n.squeeze(0)             # (B, H)

        evidence = F.softplus(self.fc(h))
        alpha = evidence + 1           # Dirichlet parameters
        return alpha


# ---------- EVIDENTIAL LOSS ----------
def evidential_loss(alpha, target):
    S = torch.sum(alpha, dim=1, keepdim=True)
    one_hot = F.one_hot(target, num_classes=2).float()
    loss = torch.sum(
        one_hot * (torch.digamma(S) - torch.digamma(alpha)),
        dim=1
    )
    return loss.mean()

# ---------- ACCURACY ----------
def compute_accuracy(alpha, targets):
    probs = alpha / alpha.sum(dim=1, keepdim=True)
    preds = probs.argmax(dim=1)
    return (preds == targets).float().mean().item()

# ---------- EARLY STOPPING ----------
class EarlyStopping:
    def __init__(self, patience=3):
        self.best = float("inf")
        self.counter = 0
        self.patience = patience
        self.stop = False

    def step(self, val_loss):
        if val_loss < self.best:
            self.best = val_loss
            self.counter = 0
            return True
        self.counter += 1
        if self.counter >= self.patience:
            self.stop = True
        return False

# ---------- TRAIN ONE MODEL ----------
def train_single_lstm(seed):
    print(f"\nTraining LSTM model with seed {seed}")
    set_seed(seed)

    model = EvidentialLSTM(VOCAB_SIZE).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    stopper = EarlyStopping(patience=3)

    save_path = os.path.join(
        RESULT_DIR, f"lstm_evidential_seed_{seed}.pt"
    )

    for epoch in range(30):
        # ----- TRAIN -----
        model.train()
        train_loss, train_acc = 0, 0

        for x, y in train_loader:
            optimizer.zero_grad()
            alpha = model(x)
            loss = evidential_loss(alpha, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_acc  += compute_accuracy(alpha, y)

        train_loss /= len(train_loader)
        train_acc  /= len(train_loader)

        # ----- VALIDATION -----
        model.eval()
        val_loss, val_acc = 0, 0

        with torch.no_grad():
            for x, y in dev_loader:
                alpha = model(x)
                val_loss += evidential_loss(alpha, y).item()
                val_acc  += compute_accuracy(alpha, y)

        val_loss /= len(dev_loader)
        val_acc  /= len(dev_loader)

        print(
            f"Epoch {epoch+1:02d} | "
            f"Train Loss {train_loss:.4f}, Acc {train_acc:.4f} | "
            f"Val Loss {val_loss:.4f}, Acc {val_acc:.4f}"
        )

        if stopper.step(val_loss):
            torch.save(model.state_dict(), save_path)
            print("   ✓ Best model saved")

        if stopper.stop:
            print("   ⛔ Early stopping")
            break

    return save_path


# ---------- DEEP ENSEMBLE ----------
SEEDS = [1, 7, 21, 42, 100]

lstm_paths = []
for seed in SEEDS:
    path = train_single_lstm(seed)
    lstm_paths.append(path)


print("\nAll ensemble models saved:")
for p in model_paths:
    print(p)


Train label distribution:
labels
0    9798
1    2259
Name: count, dtype: int64
Train size: 12057
Dev size: 3015

Training LSTM model with seed 1
Epoch 01 | Train Loss 0.5356, Acc 0.8078 | Val Loss 0.5164, Acc 0.8054
   ✓ Best model saved
Epoch 02 | Train Loss 0.5025, Acc 0.8127 | Val Loss 0.5101, Acc 0.8054
   ✓ Best model saved
Epoch 03 | Train Loss 0.4978, Acc 0.8127 | Val Loss 0.5084, Acc 0.8054
   ✓ Best model saved
Epoch 04 | Train Loss 0.4959, Acc 0.8126 | Val Loss 0.5054, Acc 0.8051
   ✓ Best model saved
Epoch 05 | Train Loss 0.4931, Acc 0.8138 | Val Loss 0.5034, Acc 0.8051
   ✓ Best model saved
Epoch 06 | Train Loss 0.4919, Acc 0.8146 | Val Loss 0.5033, Acc 0.8054
   ✓ Best model saved
Epoch 07 | Train Loss 0.4903, Acc 0.8146 | Val Loss 0.5024, Acc 0.8061
   ✓ Best model saved
Epoch 08 | Train Loss 0.4896, Acc 0.8150 | Val Loss 0.5008, Acc 0.8061
   ✓ Best model saved
Epoch 09 | Train Loss 0.4885, Acc 0.8152 | Val Loss 0.4998, Acc 0.8067
   ✓ Best model saved
Epoch 10 | Train L

NameError: name 'model_paths' is not defined

In [3]:
ensemble_models = []

for seed in SEEDS:
    model = EvidentialLSTM(VOCAB_SIZE).to(DEVICE)
    path = os.path.join(RESULT_DIR, f"lstm_evidential_seed_{seed}.pt")
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    ensemble_models.append(model)

print(f"Loaded {len(ensemble_models)} LSTM ensemble models")


Loaded 5 LSTM ensemble models


In [4]:
# ---------- UNCERTAINTY INFERENCE ----------
all_preds = []
all_labels = []

all_aleatoric = []
all_epistemic = []
all_confidence = []

with torch.no_grad():
    for x, y in dev_loader:
        x = x.to(DEVICE)

        probs_list = []
        aleatoric_list = []

        for model in ensemble_models:
            alpha = model(x)
            S = alpha.sum(dim=1, keepdim=True)

            probs = alpha / S                      # class probabilities
            aleatoric = 2.0 / S.squeeze(1)         # aleatoric uncertainty

            probs_list.append(probs)
            aleatoric_list.append(aleatoric)

        probs_stack = torch.stack(probs_list)      # (M, B, C)
        aleatoric_stack = torch.stack(aleatoric_list)

        mean_probs = probs_stack.mean(dim=0)       # ensemble mean
        epistemic = probs_stack.var(dim=0).mean(dim=1)

        preds = mean_probs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())
        all_confidence.extend(mean_probs.max(dim=1)[0].cpu().numpy())
        all_aleatoric.extend(aleatoric_stack.mean(dim=0).cpu().numpy())
        all_epistemic.extend(epistemic.cpu().numpy())


In [5]:
import numpy as np

print("Mean confidence:", np.mean(all_confidence))
print("Mean aleatoric uncertainty:", np.mean(all_aleatoric))
print("Mean epistemic uncertainty:", np.mean(all_epistemic))

print("Aleatoric min/max:", np.min(all_aleatoric), np.max(all_aleatoric))
print("Epistemic min/max:", np.min(all_epistemic), np.max(all_epistemic))


Mean confidence: 0.80124086
Mean aleatoric uncertainty: 0.06924397
Mean epistemic uncertainty: 0.014253233
Aleatoric min/max: 0.056356627 0.41339597
Epistemic min/max: 3.261126e-06 0.08419172


In [6]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(all_labels, all_preds)
print("Ensemble Accuracy (DEV):", acc)


Ensemble Accuracy (DEV): 0.8281923714759536


In [7]:
threshold = np.percentile(all_epistemic, 75)   # reject top 25% uncertain

accepted_idx = np.where(np.array(all_epistemic) < threshold)[0]

filtered_acc = accuracy_score(
    np.array(all_labels)[accepted_idx],
    np.array(all_preds)[accepted_idx]
)

print("Accuracy after rejecting uncertain samples:", filtered_acc)
print("Rejected samples:", len(all_labels) - len(accepted_idx))


Accuracy after rejecting uncertain samples: 0.8620079610791685
Rejected samples: 754


In [4]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [5]:
RESULT_DIR = r"D:\PROJECT -MTech\S4\results\evidential_deep\lstm"
# ==========================================
# LOAD ENSEMBLE MODELS
# ==========================================

SEEDS = [1, 7, 21, 42, 100]

lstm_paths = [
    os.path.join(RESULT_DIR, f"lstm_evidential_seed_{seed}.pt")
    for seed in SEEDS
]

print("Loaded model paths:")
for p in lstm_paths:
    print(p)


Loaded model paths:
D:\PROJECT -MTech\S4\results\evidential_deep\lstm\lstm_evidential_seed_1.pt
D:\PROJECT -MTech\S4\results\evidential_deep\lstm\lstm_evidential_seed_7.pt
D:\PROJECT -MTech\S4\results\evidential_deep\lstm\lstm_evidential_seed_21.pt
D:\PROJECT -MTech\S4\results\evidential_deep\lstm\lstm_evidential_seed_42.pt
D:\PROJECT -MTech\S4\results\evidential_deep\lstm\lstm_evidential_seed_100.pt


In [6]:
# ==========================================
# ENSEMBLE PREDICTION (SAFE VERSION)
# ==========================================

def ensemble_predict_with_index(models, loader):
    all_probs = []
    all_labels = []
    all_indices = []

    sample_index = 0

    with torch.no_grad():
        for x, y in loader:
            batch_size = x.size(0)

            batch_probs = []
            for model in models:
                alpha = model(x)
                probs = alpha / alpha.sum(dim=1, keepdim=True)
                batch_probs.append(probs)

            mean_probs = torch.stack(batch_probs).mean(dim=0)

            all_probs.append(mean_probs)
            all_labels.append(y)

            # track dataset indices correctly
            indices = list(range(sample_index, sample_index + batch_size))
            all_indices.extend(indices)

            sample_index += batch_size

    all_probs = torch.cat(all_probs)
    all_labels = torch.cat(all_labels)

    return all_probs, all_labels, all_indices


# Run prediction
probs, labels, indices = ensemble_predict_with_index(
    ensemble_models,
    dev_loader
)

confidence, preds = torch.max(probs, dim=1)

# Mask correct predictions
correct_mask = preds == labels

correct_indices = torch.tensor(indices)[correct_mask]
correct_conf = confidence[correct_mask]
correct_preds = preds[correct_mask]

# Sort by highest confidence
sorted_idx = torch.argsort(correct_conf, descending=True)

TOP_K = 10

print("\n🔥 Top High Confidence Correct Examples 🔥\n")

for i in sorted_idx[:TOP_K]:
    dataset_index = correct_indices[i].item()

    text = dev_df.iloc[dataset_index]["Text"]
    conf = correct_conf[i].item()
    pred = correct_preds[i].item()

    label_name = "Sarcastic" if pred == 1 else "Non-Sarcastic"

    print("Text:", text)
    print("Prediction:", label_name)
    print("Confidence:", round(conf, 4))
    print("-" * 90)


NameError: name 'ensemble_models' is not defined

In [7]:
SEEDS = [1, 7, 21, 42, 100]

lstm_paths = [
    os.path.join(RESULT_DIR, f"lstm_evidential_seed_{seed}.pt")
    for seed in SEEDS
]


In [8]:
def load_ensemble(paths):
    models = []
    for path in paths:
        model = EvidentialLSTM(VOCAB_SIZE).to(DEVICE)
        model.load_state_dict(torch.load(path, map_location=DEVICE))
        model.eval()
        models.append(model)
    return models


In [9]:
ensemble_models = load_ensemble(lstm_paths)

print("Ensemble loaded:", len(ensemble_models), "models")


NameError: name 'EvidentialLSTM' is not defined

In [10]:
# ==========================================================
# FULL ENSEMBLE EVALUATION BLOCK (RUN AFTER RESTART)
# ==========================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter

DEVICE = torch.device("cpu")

# ---------- PATHS ----------
DATA_DIR = r"D:\PROJECT -MTech\Datasets_SARCASM\MALAYALAM"
RESULT_DIR = r"D:\PROJECT -MTech\S4\results\evidential_deep\lstm"

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_mal_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_mal_dev.csv")

# ---------- LOAD DATA ----------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)

train_df['labels'] = train_df['labels'].str.strip().str.lower()
dev_df['labels']   = dev_df['labels'].str.strip().str.lower()

label_map = {"non-sarcastic": 0, "sarcastic": 1}
train_df['labels'] = train_df['labels'].map(label_map)
dev_df['labels']   = dev_df['labels'].map(label_map)

train_df = train_df.dropna(subset=['labels'])
dev_df   = dev_df.dropna(subset=['labels'])

train_df['labels'] = train_df['labels'].astype(int)
dev_df['labels']   = dev_df['labels'].astype(int)

# ---------- BUILD VOCAB ----------
MAX_VOCAB = 20000
MAX_LEN = 50

def build_vocab(texts):
    counter = Counter()
    for t in texts:
        counter.update(str(t).split())
    vocab = {w: i+2 for i,(w,_) in enumerate(counter.most_common(MAX_VOCAB))}
    vocab["<PAD>"] = 0
    vocab["<UNK>"] = 1
    return vocab

vocab = build_vocab(train_df['Text'])
VOCAB_SIZE = len(vocab)

def encode(text):
    tokens = str(text).split()
    ids = [vocab.get(tok, 1) for tok in tokens][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

# ---------- DATASET ----------
class SarcasmDataset(Dataset):
    def __init__(self, df):
        self.texts = df['Text'].values
        self.labels = df['labels'].values

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(encode(self.texts[idx]), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

dev_loader = DataLoader(SarcasmDataset(dev_df), batch_size=32, shuffle=False)

# ---------- MODEL ----------
class EvidentialLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        h = h_n.squeeze(0)
        evidence = F.softplus(self.fc(h))
        alpha = evidence + 1
        return alpha

# ---------- LOAD ENSEMBLE ----------
SEEDS = [1, 7, 21, 42, 100]

lstm_paths = [
    os.path.join(RESULT_DIR, f"lstm_evidential_seed_{seed}.pt")
    for seed in SEEDS
]

def load_ensemble(paths):
    models = []
    for path in paths:
        model = EvidentialLSTM(VOCAB_SIZE).to(DEVICE)
        model.load_state_dict(torch.load(path, map_location=DEVICE))
        model.eval()
        models.append(model)
    return models

ensemble_models = load_ensemble(lstm_paths)

print("Loaded Ensemble Models:", len(ensemble_models))

# ---------- ENSEMBLE PREDICTION ----------
all_probs = []
all_labels = []
all_texts = []

with torch.no_grad():
    for x, y in dev_loader:
        batch_probs = []
        for model in ensemble_models:
            alpha = model(x)
            probs = alpha / alpha.sum(dim=1, keepdim=True)
            batch_probs.append(probs)

        mean_probs = torch.stack(batch_probs).mean(dim=0)

        all_probs.append(mean_probs)
        all_labels.append(y)

all_probs = torch.cat(all_probs)
all_labels = torch.cat(all_labels)

confidence, preds = torch.max(all_probs, dim=1)

# ---------- METRICS ----------
y_true = all_labels.numpy()
y_pred = preds.numpy()

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("\n===== ENSEMBLE PERFORMANCE =====")
print("Accuracy :", round(acc,4))
print("Precision:", round(prec,4))
print("Recall   :", round(rec,4))
print("F1 Score :", round(f1,4))

# ---------- HIGH CONFIDENCE CORRECT ----------
correct_mask = preds == all_labels
correct_conf = confidence[correct_mask]
correct_indices = torch.arange(len(dev_df))[correct_mask]

sorted_idx = torch.argsort(correct_conf, descending=True)

TOP_K = 10

print("\n🔥 Top High Confidence Correct Examples 🔥\n")

for i in sorted_idx[:TOP_K]:
    idx = correct_indices[i].item()
    text = dev_df.iloc[idx]["Text"]
    conf = correct_conf[i].item()
    label_name = "Sarcastic" if y_pred[idx] == 1 else "Non-Sarcastic"

    print("Text:", text)
    print("Prediction:", label_name)
    print("Confidence:", round(conf,4))
    print("-" * 90)


Loaded Ensemble Models: 5

===== ENSEMBLE PERFORMANCE =====
Accuracy : 0.8282
Precision: 0.8365
Recall   : 0.148
F1 Score : 0.2514

🔥 Top High Confidence Correct Examples 🔥

Text: Uyiril Thodum Thalir  Viralavane Nee … Arike Nadakkane Alayum  Chudu Kattininu Koottinayay Naamoru Naal… Kinakkudilil  Chennanayu, Miru Nilavalayay  Aarum….Kaanaa… Hridayatharamathil  Uruki Naamannaarum Kelkkaa…Pranayajalakadha Palavuru Parayumo?  Uyiril Thodum Kulir Viralaayidaam Njan Arike Nadannidaam Alayum Chudu Kattininu Koottinayay  Naamoru Naal Kinakkudilil Chennanayu, Miru Nilavalayay  Aarum…Kaanaa… Hridayatharamathil Uruki Naamannaarum Kelkkaa….Pranayajalakadha Palavuru Parayumo?  Vazhiyorangal Thorum Thanalayee Padarchilla Nee Kudayay Nivarnnu Nee Novaaraathe Thoraathe Peyke Thuzhayolangal Pol Nin Kadavathonnu Njan Thottu Melle  Kaattee Chillayithil Veeshane Kaare Ilayithil Peyyane Melle Theeramithilolangalolangalay Nee Varoo….  Uyiril Thalodeedum Uyirayidum Naam…. Naamorunaal Kinakkadalil Chennayum

In [12]:
# ==========================================================
# FULL ENSEMBLE EVALUATION BLOCK (WITH ACTUAL CLASS)
# ==========================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter

DEVICE = torch.device("cpu")

# ---------- PATHS ----------
DATA_DIR = r"D:\PROJECT -MTech\Datasets_SARCASM\MALAYALAM"
RESULT_DIR = r"D:\PROJECT -MTech\S4\results\evidential_deep\lstm"

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_mal_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_mal_dev.csv")

# ---------- LOAD DATA ----------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)

train_df['labels'] = train_df['labels'].str.strip().str.lower()
dev_df['labels']   = dev_df['labels'].str.strip().str.lower()

label_map = {"non-sarcastic": 0, "sarcastic": 1}
train_df['labels'] = train_df['labels'].map(label_map)
dev_df['labels']   = dev_df['labels'].map(label_map)

train_df = train_df.dropna(subset=['labels'])
dev_df   = dev_df.dropna(subset=['labels'])

train_df['labels'] = train_df['labels'].astype(int)
dev_df['labels']   = dev_df['labels'].astype(int)

# ---------- BUILD VOCAB ----------
MAX_VOCAB = 20000
MAX_LEN = 50

def build_vocab(texts):
    counter = Counter()
    for t in texts:
        counter.update(str(t).split())
    vocab = {w: i+2 for i,(w,_) in enumerate(counter.most_common(MAX_VOCAB))}
    vocab["<PAD>"] = 0
    vocab["<UNK>"] = 1
    return vocab

vocab = build_vocab(train_df['Text'])
VOCAB_SIZE = len(vocab)

def encode(text):
    tokens = str(text).split()
    ids = [vocab.get(tok, 1) for tok in tokens][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

# ---------- DATASET ----------
class SarcasmDataset(Dataset):
    def __init__(self, df):
        self.texts = df['Text'].values
        self.labels = df['labels'].values

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(encode(self.texts[idx]), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

dev_loader = DataLoader(SarcasmDataset(dev_df), batch_size=32, shuffle=False)

# ---------- MODEL ----------
class EvidentialLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        h = h_n.squeeze(0)
        evidence = F.softplus(self.fc(h))
        alpha = evidence + 1
        return alpha

# ---------- LOAD ENSEMBLE ----------
SEEDS = [1, 7, 21, 42, 100]

lstm_paths = [
    os.path.join(RESULT_DIR, f"lstm_evidential_seed_{seed}.pt")
    for seed in SEEDS
]

def load_ensemble(paths):
    models = []
    for path in paths:
        model = EvidentialLSTM(VOCAB_SIZE).to(DEVICE)
        model.load_state_dict(torch.load(path, map_location=DEVICE))
        model.eval()
        models.append(model)
    return models

ensemble_models = load_ensemble(lstm_paths)

print("Loaded Ensemble Models:", len(ensemble_models))

# ---------- ENSEMBLE PREDICTION ----------
all_probs = []
all_labels = []

with torch.no_grad():
    for x, y in dev_loader:
        batch_probs = []
        for model in ensemble_models:
            alpha = model(x)
            probs = alpha / alpha.sum(dim=1, keepdim=True)
            batch_probs.append(probs)

        mean_probs = torch.stack(batch_probs).mean(dim=0)

        all_probs.append(mean_probs)
        all_labels.append(y)

all_probs = torch.cat(all_probs)
all_labels = torch.cat(all_labels)

confidence, preds = torch.max(all_probs, dim=1)

# ---------- METRICS ----------
y_true = all_labels.numpy()
y_pred = preds.numpy()

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("\n===== ENSEMBLE PERFORMANCE =====")
print("Accuracy :", round(acc,4))
print("Precision:", round(prec,4))
print("Recall   :", round(rec,4))
print("F1 Score :", round(f1,4))

# ---------- HIGH CONFIDENCE CORRECT ----------
correct_mask = preds == all_labels
correct_conf = confidence[correct_mask]
correct_indices = torch.arange(len(dev_df))[correct_mask]

sorted_idx = torch.argsort(correct_conf, descending=True)

TOP_K = 50

print("\n🔥 Top High Confidence Correct Examples 🔥\n")

for i in sorted_idx[:TOP_K]:
    idx = correct_indices[i].item()

    text = dev_df.iloc[idx]["Text"]
    pred_class = "Sarcastic" if y_pred[idx] == 1 else "Non-Sarcastic"
    actual_class = "Sarcastic" if y_true[idx] == 1 else "Non-Sarcastic"
    conf = correct_conf[i].item()

    print("Text:", text)
    print("Predicted Class:", pred_class)
    print("Actual Class   :", actual_class)
    print("Confidence     :", round(conf,4))
    print("-" * 90)


Loaded Ensemble Models: 5

===== ENSEMBLE PERFORMANCE =====
Accuracy : 0.8282
Precision: 0.8365
Recall   : 0.148
F1 Score : 0.2514

🔥 Top High Confidence Correct Examples 🔥

Text: Uyiril Thodum Thalir  Viralavane Nee … Arike Nadakkane Alayum  Chudu Kattininu Koottinayay Naamoru Naal… Kinakkudilil  Chennanayu, Miru Nilavalayay  Aarum….Kaanaa… Hridayatharamathil  Uruki Naamannaarum Kelkkaa…Pranayajalakadha Palavuru Parayumo?  Uyiril Thodum Kulir Viralaayidaam Njan Arike Nadannidaam Alayum Chudu Kattininu Koottinayay  Naamoru Naal Kinakkudilil Chennanayu, Miru Nilavalayay  Aarum…Kaanaa… Hridayatharamathil Uruki Naamannaarum Kelkkaa….Pranayajalakadha Palavuru Parayumo?  Vazhiyorangal Thorum Thanalayee Padarchilla Nee Kudayay Nivarnnu Nee Novaaraathe Thoraathe Peyke Thuzhayolangal Pol Nin Kadavathonnu Njan Thottu Melle  Kaattee Chillayithil Veeshane Kaare Ilayithil Peyyane Melle Theeramithilolangalolangalay Nee Varoo….  Uyiril Thalodeedum Uyirayidum Naam…. Naamorunaal Kinakkadalil Chennayum

In [1]:
# ==========================================================
# FULL ENSEMBLE EVALUATION BLOCK (WITH ACTUAL CLASS)
# ==========================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter

DEVICE = torch.device("cpu")

# ---------- PATHS ----------
DATA_DIR = r"D:\PROJECT -MTech\Datasets_SARCASM\MALAYALAM"
RESULT_DIR = r"D:\PROJECT -MTech\S4\results\evidential_deep\lstm"

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_mal_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_mal_dev.csv")

# ---------- LOAD DATA ----------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)

train_df['labels'] = train_df['labels'].str.strip().str.lower()
dev_df['labels']   = dev_df['labels'].str.strip().str.lower()

label_map = {"non-sarcastic": 0, "sarcastic": 1}
train_df['labels'] = train_df['labels'].map(label_map)
dev_df['labels']   = dev_df['labels'].map(label_map)

train_df = train_df.dropna(subset=['labels'])
dev_df   = dev_df.dropna(subset=['labels'])

train_df['labels'] = train_df['labels'].astype(int)
dev_df['labels']   = dev_df['labels'].astype(int)

# ---------- BUILD VOCAB ----------
MAX_VOCAB = 20000
MAX_LEN = 50

def build_vocab(texts):
    counter = Counter()
    for t in texts:
        counter.update(str(t).split())
    vocab = {w: i+2 for i,(w,_) in enumerate(counter.most_common(MAX_VOCAB))}
    vocab["<PAD>"] = 0
    vocab["<UNK>"] = 1
    return vocab

vocab = build_vocab(train_df['Text'])
VOCAB_SIZE = len(vocab)

def encode(text):
    tokens = str(text).split()
    ids = [vocab.get(tok, 1) for tok in tokens][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

# ---------- DATASET ----------
class SarcasmDataset(Dataset):
    def __init__(self, df):
        self.texts = df['Text'].values
        self.labels = df['labels'].values

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(encode(self.texts[idx]), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

dev_loader = DataLoader(SarcasmDataset(dev_df), batch_size=32, shuffle=False)

# ---------- MODEL ----------
class EvidentialLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        h = h_n.squeeze(0)
        evidence = F.softplus(self.fc(h))
        alpha = evidence + 1
        return alpha

# ---------- LOAD ENSEMBLE ----------
SEEDS = [1, 7, 21, 42, 100]

lstm_paths = [
    os.path.join(RESULT_DIR, f"lstm_evidential_seed_{seed}.pt")
    for seed in SEEDS
]

def load_ensemble(paths):
    models = []
    for path in paths:
        model = EvidentialLSTM(VOCAB_SIZE).to(DEVICE)
        model.load_state_dict(torch.load(path, map_location=DEVICE))
        model.eval()
        models.append(model)
    return models

ensemble_models = load_ensemble(lstm_paths)

print("Loaded Ensemble Models:", len(ensemble_models))

# ---------- ENSEMBLE PREDICTION ----------
all_probs = []
all_labels = []

with torch.no_grad():
    for x, y in dev_loader:
        batch_probs = []
        for model in ensemble_models:
            alpha = model(x)
            probs = alpha / alpha.sum(dim=1, keepdim=True)
            batch_probs.append(probs)

        mean_probs = torch.stack(batch_probs).mean(dim=0)

        all_probs.append(mean_probs)
        all_labels.append(y)

all_probs = torch.cat(all_probs)
all_labels = torch.cat(all_labels)

confidence, preds = torch.max(all_probs, dim=1)

# ---------- METRICS ----------
y_true = all_labels.numpy()
y_pred = preds.numpy()

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("\n===== ENSEMBLE PERFORMANCE =====")
print("Accuracy :", round(acc,4))
print("Precision:", round(prec,4))
print("Recall   :", round(rec,4))
print("F1 Score :", round(f1,4))

# ---------- HIGH CONFIDENCE CORRECT (Malayalam Only) ----------

correct_mask = preds == all_labels
correct_conf = confidence[correct_mask]
correct_indices = torch.arange(len(dev_df))[correct_mask]

# Filter only Malayalam text
mal_indices = []

for i in range(len(correct_indices)):
    idx = correct_indices[i].item()
    text = dev_df.iloc[idx]["Text"]

    if contains_malayalam(text):
        mal_indices.append(i)

mal_indices = torch.tensor(mal_indices)

mal_correct_conf = correct_conf[mal_indices]
mal_correct_indices = correct_indices[mal_indices]

sorted_idx = torch.argsort(mal_correct_conf, descending=True)

TOP_K = 50

print("\n🔥 Top High Confidence Correct MALAYALAM Examples 🔥\n")

for i in sorted_idx[:TOP_K]:
    idx = mal_correct_indices[i].item()

    text = dev_df.iloc[idx]["Text"]
    pred_class = "Sarcastic" if y_pred[idx] == 1 else "Non-Sarcastic"
    actual_class = "Sarcastic" if y_true[idx] == 1 else "Non-Sarcastic"
    conf = confidence[idx].item()

    print("Text:", text)
    print("Predicted Class:", pred_class)
    print("Actual Class   :", actual_class)
    print("Confidence     :", round(conf,4))
    print("-" * 90)

Loaded Ensemble Models: 5

===== ENSEMBLE PERFORMANCE =====
Accuracy : 0.8282
Precision: 0.8365
Recall   : 0.148
F1 Score : 0.2514


NameError: name 'contains_malayalam' is not defined

In [13]:
# ==========================================================
# LOW CONFIDENCE ANALYSIS
# ==========================================================

# Confidence already computed earlier:
# confidence, preds, y_true, y_pred

# Convert to tensors if needed
confidence_tensor = confidence
preds_tensor = preds
labels_tensor = all_labels

# -----------------------------
# 1️⃣ LOW CONFIDENCE CORRECT
# -----------------------------
correct_mask = preds_tensor == labels_tensor
low_conf_correct = confidence_tensor[correct_mask]
low_conf_indices = torch.arange(len(dev_df))[correct_mask]

# sort ascending (lowest confidence first)
sorted_low_correct = torch.argsort(low_conf_correct)

TOP_K = 5

print("\n❄️ LOW Confidence BUT Correct Predictions ❄️\n")

for i in sorted_low_correct[:TOP_K]:
    idx = low_conf_indices[i].item()

    text = dev_df.iloc[idx]["Text"]
    pred_class = "Sarcastic" if y_pred[idx] == 1 else "Non-Sarcastic"
    actual_class = "Sarcastic" if y_true[idx] == 1 else "Non-Sarcastic"
    conf = low_conf_correct[i].item()

    print("Text:", text)
    print("Predicted Class:", pred_class)
    print("Actual Class   :", actual_class)
    print("Confidence     :", round(conf,4))
    print("-" * 90)


# -----------------------------
# 2️⃣ LOW CONFIDENCE WRONG
# -----------------------------
wrong_mask = preds_tensor != labels_tensor
low_conf_wrong = confidence_tensor[wrong_mask]
wrong_indices = torch.arange(len(dev_df))[wrong_mask]

sorted_low_wrong = torch.argsort(low_conf_wrong)

print("\n⚠️ LOW Confidence AND Wrong Predictions ⚠️\n")

for i in sorted_low_wrong[:TOP_K]:
    idx = wrong_indices[i].item()

    text = dev_df.iloc[idx]["Text"]
    pred_class = "Sarcastic" if y_pred[idx] == 1 else "Non-Sarcastic"
    actual_class = "Sarcastic" if y_true[idx] == 1 else "Non-Sarcastic"
    conf = low_conf_wrong[i].item()

    print("Text:", text)
    print("Predicted Class:", pred_class)
    print("Actual Class   :", actual_class)
    print("Confidence     :", round(conf,4))
    print("-" * 90)



❄️ LOW Confidence BUT Correct Predictions ❄️

Text: ഇതെന്ത് ...പാർട്ടി പ്രസംഗം പോലെ ഉണ്ടല്ലോ..dialogue.. Delivery
Predicted Class: Sarcastic
Actual Class   : Sarcastic
Confidence     : 0.5004
------------------------------------------------------------------------------------------
Text: ആദ്യം ജിമ്മിക്കി കമ്മൽ ഇപ്പൊ മുക്കുറ്റി കാണാനില്ല   നാട് മൊത്തം മോഷണം ആണല്ലോ 🤔🤔🤔
Predicted Class: Sarcastic
Actual Class   : Sarcastic
Confidence     : 0.5007
------------------------------------------------------------------------------------------
Text: Kanendathu ellam naatukaru kanditund .ini mookuthi kandillelum nooo prblm.
Predicted Class: Sarcastic
Actual Class   : Sarcastic
Confidence     : 0.5007
------------------------------------------------------------------------------------------
Text: വലിയ സുഖം ഒന്നും തോന്നാത്ത സംഗീതം. ഇത്രയും ദൃശ്യ ഭംഗി ഒക്കെ ചിത്രീകരിച്ചപ്പോൾ സാമാന്യം ഒരു നല്ല പാട്ടു ആകാമായിരുന്നു.. ആ ദീപക് ദേവിന് കൊടുത്തിരുന്നെങ്കിൽ മൂപ്പര് bore ആക്കില്ലായിരുന്നു.
Predicted Class: S

In [14]:
# ==========================================================
# MODEL FAILURE ANALYSIS
# ==========================================================

# preds, confidence, y_true already computed earlier

preds_tensor = preds
labels_tensor = all_labels
confidence_tensor = confidence

# -----------------------------------
# 1️⃣ ALL WRONG PREDICTIONS
# -----------------------------------
wrong_mask = preds_tensor != labels_tensor
wrong_indices = torch.arange(len(dev_df))[wrong_mask]
wrong_conf = confidence_tensor[wrong_mask]

print("\n❌ TOTAL FAILURES:", wrong_mask.sum().item())

TOP_K = 10

print("\n🔎 SAMPLE WRONG PREDICTIONS 🔎\n")

for i in range(min(TOP_K, len(wrong_indices))):
    idx = wrong_indices[i].item()

    text = dev_df.iloc[idx]["Text"]
    pred_class = "Sarcastic" if y_pred[idx] == 1 else "Non-Sarcastic"
    actual_class = "Sarcastic" if y_true[idx] == 1 else "Non-Sarcastic"
    conf = wrong_conf[i].item()

    print("Text:", text)
    print("Predicted Class:", pred_class)
    print("Actual Class   :", actual_class)
    print("Confidence     :", round(conf,4))
    print("-" * 90)


# -----------------------------------
# 2️⃣ HIGH CONFIDENCE WRONG (DANGEROUS)
# -----------------------------------
sorted_high_wrong = torch.argsort(wrong_conf, descending=True)

print("\n🔥 HIGH CONFIDENCE WRONG (Overconfident Errors) 🔥\n")

for i in sorted_high_wrong[:5]:
    idx = wrong_indices[i].item()

    text = dev_df.iloc[idx]["Text"]
    pred_class = "Sarcastic" if y_pred[idx] == 1 else "Non-Sarcastic"
    actual_class = "Sarcastic" if y_true[idx] == 1 else "Non-Sarcastic"
    conf = wrong_conf[i].item()

    print("Text:", text)
    print("Predicted Class:", pred_class)
    print("Actual Class   :", actual_class)
    print("Confidence     :", round(conf,4))
    print("-" * 90)


# -----------------------------------
# 3️⃣ LOW CONFIDENCE WRONG
# -----------------------------------
sorted_low_wrong = torch.argsort(wrong_conf)

print("\n❄️ LOW CONFIDENCE WRONG (Uncertain Errors) ❄️\n")

for i in sorted_low_wrong[:5]:
    idx = wrong_indices[i].item()

    text = dev_df.iloc[idx]["Text"]
    pred_class = "Sarcastic" if y_pred[idx] == 1 else "Non-Sarcastic"
    actual_class = "Sarcastic" if y_true[idx] == 1 else "Non-Sarcastic"
    conf = wrong_conf[i].item()

    print("Text:", text)
    print("Predicted Class:", pred_class)
    print("Actual Class   :", actual_class)
    print("Confidence     :", round(conf,4))
    print("-" * 90)



❌ TOTAL FAILURES: 518

🔎 SAMPLE WRONG PREDICTIONS 🔎

Text: ശ്ശെ ...!! അനു ചേച്ചി വിത്ത് മുലക്കച്ച പ്രതീക്ഷിച്ചു
Predicted Class: Non-Sarcastic
Actual Class   : Sarcastic
Confidence     : 0.5132
------------------------------------------------------------------------------------------
Text: ഒരു 120 പോയി കിട്ടി എജ്ജാതി ട്രൈലെർ
Predicted Class: Non-Sarcastic
Actual Class   : Sarcastic
Confidence     : 0.5099
------------------------------------------------------------------------------------------
Text: ലാലേട്ടന്റെ അനിയൻ ആയി ജാക്ക് സ്പാർറോ ഉണ്ട് ഹിഹിഹി
Predicted Class: Non-Sarcastic
Actual Class   : Sarcastic
Confidence     : 0.673
------------------------------------------------------------------------------------------
Text: eth konathile trailer ado !! kandit oru padakam feel akunuu!!
Predicted Class: Non-Sarcastic
Actual Class   : Sarcastic
Confidence     : 0.9328
------------------------------------------------------------------------------------------
Text: സംഭാഷണം ശരിയല്ല. അതിനു ന

In [2]:
# ==========================================================
# FULL ENSEMBLE EVALUATION BLOCK (WITH ACTUAL CLASS)
# ==========================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter

DEVICE = torch.device("cpu")

# ---------- PATHS ----------
DATA_DIR = r"D:\PROJECT -MTech\Datasets_SARCASM\MALAYALAM"
RESULT_DIR = r"D:\PROJECT -MTech\S4\results\evidential_deep\lstm"

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_mal_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_mal_dev.csv")

# ---------- LOAD DATA ----------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)

train_df['labels'] = train_df['labels'].str.strip().str.lower()
dev_df['labels']   = dev_df['labels'].str.strip().str.lower()

label_map = {"non-sarcastic": 0, "sarcastic": 1}
train_df['labels'] = train_df['labels'].map(label_map)
dev_df['labels']   = dev_df['labels'].map(label_map)

train_df = train_df.dropna(subset=['labels'])
dev_df   = dev_df.dropna(subset=['labels'])

train_df['labels'] = train_df['labels'].astype(int)
dev_df['labels']   = dev_df['labels'].astype(int)

# ---------- BUILD VOCAB ----------
MAX_VOCAB = 20000
MAX_LEN = 50

def build_vocab(texts):
    counter = Counter()
    for t in texts:
        counter.update(str(t).split())
    vocab = {w: i+2 for i,(w,_) in enumerate(counter.most_common(MAX_VOCAB))}
    vocab["<PAD>"] = 0
    vocab["<UNK>"] = 1
    return vocab

vocab = build_vocab(train_df['Text'])
VOCAB_SIZE = len(vocab)

def encode(text):
    tokens = str(text).split()
    ids = [vocab.get(tok, 1) for tok in tokens][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

# ---------- DATASET ----------
class SarcasmDataset(Dataset):
    def __init__(self, df):
        self.texts = df['Text'].values
        self.labels = df['labels'].values

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(encode(self.texts[idx]), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

dev_loader = DataLoader(SarcasmDataset(dev_df), batch_size=32, shuffle=False)

# ---------- MODEL ----------
class EvidentialLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        h = h_n.squeeze(0)
        evidence = F.softplus(self.fc(h))
        alpha = evidence + 1
        return alpha

# ---------- LOAD ENSEMBLE ----------
SEEDS = [1, 7, 21, 42, 100]

lstm_paths = [
    os.path.join(RESULT_DIR, f"lstm_evidential_seed_{seed}.pt")
    for seed in SEEDS
]

def load_ensemble(paths):
    models = []
    for path in paths:
        model = EvidentialLSTM(VOCAB_SIZE).to(DEVICE)
        model.load_state_dict(torch.load(path, map_location=DEVICE))
        model.eval()
        models.append(model)
    return models

ensemble_models = load_ensemble(lstm_paths)

print("Loaded Ensemble Models:", len(ensemble_models))

# ---------- ENSEMBLE PREDICTION ----------
all_probs = []
all_labels = []

with torch.no_grad():
    for x, y in dev_loader:
        batch_probs = []
        for model in ensemble_models:
            alpha = model(x)
            probs = alpha / alpha.sum(dim=1, keepdim=True)
            batch_probs.append(probs)

        mean_probs = torch.stack(batch_probs).mean(dim=0)

        all_probs.append(mean_probs)
        all_labels.append(y)

all_probs = torch.cat(all_probs)
all_labels = torch.cat(all_labels)

confidence, preds = torch.max(all_probs, dim=1)

# ---------- METRICS ----------
y_true = all_labels.numpy()
y_pred = preds.numpy()

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("\n===== ENSEMBLE PERFORMANCE =====")
print("Accuracy :", round(acc,4))
print("Precision:", round(prec,4))
print("Recall   :", round(rec,4))
print("F1 Score :", round(f1,4))

# ---------- HIGH CONFIDENCE CORRECT (Malayalam Only) ----------
import re

# Malayalam Unicode range
malayalam_pattern = re.compile(r'[\u0D00-\u0D7F]+')

correct_mask = preds == all_labels
correct_conf = confidence[correct_mask]
correct_indices = torch.arange(len(dev_df))[correct_mask]

sorted_idx = torch.argsort(correct_conf, descending=True)

TOP_K = 50

print("\n🔥 Top High Confidence Correct Malayalam Examples 🔥\n")

count = 0

for i in sorted_idx:
    idx = correct_indices[i].item()
    text = str(dev_df.iloc[idx]["Text"])

    # ✅ Keep only Malayalam-containing sentences
    if not malayalam_pattern.search(text):
        continue

    pred_class = "Sarcastic" if y_pred[idx] == 1 else "Non-Sarcastic"
    actual_class = "Sarcastic" if y_true[idx] == 1 else "Non-Sarcastic"
    conf = correct_conf[i].item()

    print("Text:", text)
    print("Predicted Class:", pred_class)
    print("Actual Class   :", actual_class)
    print("Confidence     :", round(conf,4))
    print("-" * 90)

    count += 1
    if count >= TOP_K:
        break


Loaded Ensemble Models: 5

===== ENSEMBLE PERFORMANCE =====
Accuracy : 0.8282
Precision: 0.8365
Recall   : 0.148
F1 Score : 0.2514

🔥 Top High Confidence Correct Malayalam Examples 🔥

Text: ഉയിരിൽ തൊടും തളിർ  വിരലാവണേ നീ.. അരികേ നടക്കണേ.. അലയും  ചുടുകാറ്റിനു കൂട്ടിണയായ് നാമൊരു നാൾ കിനാക്കുടിലിൽ  ചെന്നണയുമിരുനിലാവലയായ്..  ആരും.. കാണാ… ഹൃദയതാരമതിൽ  ഉരുകി നാമന്നാരും.. കേൾക്കാ… പ്രണയാജാലകഥ  പലവുരു പറയുമോ…  ഉയിരിൽ തൊടും കുളിർ  വിരലായിടാം ഞാൻ..  അരികേ നടന്നിടാം.. അലയും  ചുടുകാറ്റിനു കൂട്ടിണയായ് നാമൊരു നാൾ കിനാക്കുടിലിൽ  ചെന്നണയുമിരുനിലാവലയായ്..  ആരും.. കാണാ… ഹൃദയതാരമതിൽ  ഉരുകി നാമന്നാരും.. കേൾക്കാ… പ്രണയാജാലകഥ  പലവുരു പറയുമോ…  വഴിയോരങ്ങൾ തോറും  തണലായീ പടർച്ചില്ല നീ… കുടയായ് നിവർന്നൂ നീ  നോവാറാതെ തോരാതെ പെയ്‌കേ.. തുഴയോളങ്ങൾ പോൽ നിൻ  കടവത്തോന്നു ഞാൻ തൊട്ടു മെല്ലെ കാറ്റേ ചില്ലയിതിൽ വീശണേ കാറേ ഇലയിതിൽ പെയ്യണേ  മെല്ലെ തീരമിതിലോളങ്ങളോളങ്ങളായി നീ വരൂ…  ഉയിരിൽ തലോടിടും  ഉയിരായിടും നാം..  നാമൊരു നാൾ കിനാക്കടലിൽ  ചെന്നണയുമിരുനിലാനദിയായ്..  ആരും.. കാണാ… ഹൃദയതാരമതിൽ  ഉരുകി നാമന്നാരും.. കേൾക്കാ… പ്രണയാജാ

In [3]:
# ==========================================================
# LOW CONFIDENCE CASES (Malayalam Only)
# ==========================================================

print("\n❄️ LOW Confidence BUT Correct Malayalam Examples ❄️\n")

# Sort ascending (lowest confidence first)
sorted_low_idx = torch.argsort(correct_conf)

count = 0

for i in sorted_low_idx:
    idx = correct_indices[i].item()
    text = str(dev_df.iloc[idx]["Text"])

    # Malayalam filter
    if not malayalam_pattern.search(text):
        continue

    pred_class = "Sarcastic" if y_pred[idx] == 1 else "Non-Sarcastic"
    actual_class = "Sarcastic" if y_true[idx] == 1 else "Non-Sarcastic"
    conf = correct_conf[i].item()

    print("Text:", text)
    print("Predicted Class:", pred_class)
    print("Actual Class   :", actual_class)
    print("Confidence     :", round(conf,4))
    print("-" * 90)

    count += 1
    if count >= 20:
        break


# ==========================================================
# LOW CONFIDENCE WRONG (Model Failed)
# ==========================================================

print("\n⚠️ LOW Confidence WRONG Malayalam Examples ⚠️\n")

wrong_mask = preds != all_labels
wrong_conf = confidence[wrong_mask]
wrong_indices = torch.arange(len(dev_df))[wrong_mask]

sorted_wrong_low = torch.argsort(wrong_conf)

count = 0

for i in sorted_wrong_low:
    idx = wrong_indices[i].item()
    text = str(dev_df.iloc[idx]["Text"])

    # Malayalam filter
    if not malayalam_pattern.search(text):
        continue

    pred_class = "Sarcastic" if y_pred[idx] == 1 else "Non-Sarcastic"
    actual_class = "Sarcastic" if y_true[idx] == 1 else "Non-Sarcastic"
    conf = wrong_conf[i].item()

    print("Text:", text)
    print("Predicted Class:", pred_class)
    print("Actual Class   :", actual_class)
    print("Confidence     :", round(conf,4))
    print("-" * 90)

    count += 1
    if count >= 20:
        break



❄️ LOW Confidence BUT Correct Malayalam Examples ❄️

Text: ഇതെന്ത് ...പാർട്ടി പ്രസംഗം പോലെ ഉണ്ടല്ലോ..dialogue.. Delivery
Predicted Class: Sarcastic
Actual Class   : Sarcastic
Confidence     : 0.5004
------------------------------------------------------------------------------------------
Text: ആദ്യം ജിമ്മിക്കി കമ്മൽ ഇപ്പൊ മുക്കുറ്റി കാണാനില്ല   നാട് മൊത്തം മോഷണം ആണല്ലോ 🤔🤔🤔
Predicted Class: Sarcastic
Actual Class   : Sarcastic
Confidence     : 0.5007
------------------------------------------------------------------------------------------
Text: വലിയ സുഖം ഒന്നും തോന്നാത്ത സംഗീതം. ഇത്രയും ദൃശ്യ ഭംഗി ഒക്കെ ചിത്രീകരിച്ചപ്പോൾ സാമാന്യം ഒരു നല്ല പാട്ടു ആകാമായിരുന്നു.. ആ ദീപക് ദേവിന് കൊടുത്തിരുന്നെങ്കിൽ മൂപ്പര് bore ആക്കില്ലായിരുന്നു.
Predicted Class: Sarcastic
Actual Class   : Sarcastic
Confidence     : 0.5016
------------------------------------------------------------------------------------------
Text: മാമാങ്കം Review പഴശ്ശിരാജ പ്രതീക്ഷിച്ച് ആരും കാണരുതേ  മാമാങ്കം എന്ന വാണിജ്യമേള.സാമൂതിര